# Ilaria RVC on Google Colab

Run **Ilaria RVC 2.4.0** on a Linux runtime with an NVIDIA GPU. Select **Runtime → Change runtime type → GPU**, then run steps **1–4** in order. The Web UI opens in the output of step 4 through Colab's built-in port viewer.

The notebook installs its own Python environment and downloads checksummed base weights. Extra embedders can be downloaded from **Other → Download embedders**. Copy anything you want to keep before deleting or disconnecting the runtime. Colab may terminate Web UI workloads on its free tier. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

## 1. Check GPU and get Ilaria RVC

In [ ]:
#@title Check GPU and get Ilaria RVC
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

WORK = Path("/content")
ROOT = WORK / "ilaria-rvc"
TOOLS = WORK / ".ilaria-colab"
PYTHON = TOOLS / "venv" / "bin" / "python"
SOURCE_URL = "https://github.com/TheStingerX/Ilaria-RVC.git"
SOURCE_REVISION = "626e28b3ab57aeaa9e50e4160dbbc9813db281f7"

def run(*args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if sys.platform != "linux" or not WORK.is_dir():
    raise RuntimeError("Open this notebook in Google Colab.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("In Runtime settings, select an NVIDIA GPU and reconnect.")
run("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")
TOOLS.mkdir(exist_ok=True)
if not ROOT.exists():
    ROOT.mkdir()
    run("git", "init", ROOT)
if not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} already exists and is not this notebook's checkout.")
if not (ROOT / "ilaria" / "ilaria_version.json").is_file():
    run("git", "-C", ROOT, "fetch", "--depth", "1", SOURCE_URL, SOURCE_REVISION)
    run("git", "-C", ROOT, "checkout", "--detach", "FETCH_HEAD")
else:
    current = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    if current != SOURCE_REVISION:
        raise RuntimeError(f"{ROOT} contains a different revision. Save your work and use a fresh runtime.")
if not (ROOT / ".env").exists():
    shutil.copyfile(ROOT / ".env.example", ROOT / ".env")
for name in ("models", "datasets", "InferOutput"):
    (ROOT / name).mkdir(exist_ok=True)
version = json.loads((ROOT / "ilaria/ilaria_version.json").read_text(encoding="utf-8"))["version"]
print(f"Ilaria RVC {version} is ready for installation.")


## 2. Install dependencies

The first installation builds the native audio dependencies. The progress card shows the current step and elapsed time.

In [ ]:
#@title Install dependencies
import time
from html import escape
from IPython.display import HTML, display
from threading import Event, Thread

def install_environment(report):
    install_marker = TOOLS / "colab-environment-cu128.ok"
    runtime_env = {**os.environ, "GRADIO_ANALYTICS_ENABLED": "False",
                   "OMP_NUM_THREADS": "2", "MPLBACKEND": "Agg"}
    report(0, "Checking system tools")
    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run("apt-get", "-q", "update")
        run("apt-get", "-qy", "install", "ffmpeg")
    for binary in ("ffmpeg", "ffprobe"):
        executable = ROOT / (binary + ".exe")
        if not executable.exists():
            executable.symlink_to(shutil.which(binary))
    if not all(shutil.which(binary) for binary in ("gcc", "g++", "make")):
        run("apt-get", "-q", "update")
        run("apt-get", "-qy", "install", "build-essential")

    if not (PYTHON.is_file() and install_marker.is_file()):
        report(1, "Preparing Python")
        bootstrap = TOOLS / "bootstrap"
        if not (bootstrap / "uv").is_dir():
            run(sys.executable, "-m", "pip", "install", "--quiet", "--target", bootstrap, "uv==0.9.10")
        uv_env = {
            **os.environ,
            "PYTHONPATH": str(bootstrap),
            "UV_PYTHON_INSTALL_DIR": str(TOOLS / "python"),
            "UV_CACHE_DIR": str(TOOLS / "uv-cache"),
        }
        if not PYTHON.is_file():
            run(sys.executable, "-m", "uv", "python", "install", "3.10", env=uv_env)
            run(sys.executable, "-m", "uv", "venv", "--python", "3.10", "--seed", TOOLS / "venv", env=uv_env)

        install_log = TOOLS / "install.log"

        def uv_install(*args):
            with install_log.open("a", encoding="utf-8") as log:
                result = subprocess.run(
                    [sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON), *args],
                    env=uv_env, stdout=log, stderr=subprocess.STDOUT,
                )
            if result.returncode:
                print(install_log.read_text(encoding="utf-8", errors="replace")[-12000:], flush=True)
                raise RuntimeError(f"uv installation failed. Full log: {install_log}")

        def pip(*args):
            with install_log.open("a", encoding="utf-8") as log:
                result = subprocess.run(
                    [str(PYTHON), "-m", "pip", "--disable-pip-version-check", *args],
                    stdout=log, stderr=subprocess.STDOUT,
                )
            if result.returncode:
                with install_log.open("rb") as log:
                    size = log.seek(0, os.SEEK_END)
                    log.seek(max(0, size - 12000))
                    print(log.read().decode("utf-8", errors="replace"), flush=True)
                raise RuntimeError(f"Dependency installation failed. Full log: {install_log}")

        pip("install", "pip==24.0", "setuptools==69.5.1", "wheel==0.43.0", "numpy==1.23.5")
        report(2, "Installing CUDA PyTorch")
        uv_install("--index", "https://download.pytorch.org/whl/cu128",
            "torch==2.10.0+cu128", "torchaudio==2.10.0+cu128")
        report(3, "Preparing native audio dependencies")
        # Build native modules separately so compiler errors reach the notebook output.
        build_env = {**os.environ, "CC": shutil.which("gcc"), "CXX": shutil.which("g++"),
                     "CPLUS_INCLUDE_PATH": os.environ.get("CPLUS_INCLUDE_PATH", ""),
                     "MAX_JOBS": "2"}
        if not build_env["CPLUS_INCLUDE_PATH"]:
            build_env.pop("CPLUS_INCLUDE_PATH")
        # HuBERT uses Fairseq's CPU extensions. Keep CUDA enabled for the app.
        # Fairseq enables CUDA extensions whenever CUDA_HOME exists, even if empty.
        build_env.pop("CUDA_HOME", None)
        build_env.update(CUDA_VISIBLE_DEVICES="", FORCE_CUDA="0")
        # The PyPI sdist omits C++ files included in the upstream release.
        fairseq_source = "https://codeload.github.com/facebookresearch/fairseq/tar.gz/refs/tags/v0.12.2"
        for package, source in (("fairseq==0.12.2", fairseq_source), ("pyworld==0.3.5", "pyworld==0.3.5")):
            report(3 if package.startswith("fairseq") else 4, f"Preparing {package}")
            name, version = package.split("==", 1)
            installed = subprocess.run(
                [str(PYTHON), "-c",
                 "from importlib.metadata import version; import sys; print(version(sys.argv[1]))",
                 name],
                capture_output=True, text=True,
            )
            if installed.returncode == 0 and installed.stdout.strip() == version:
                print(f"Reusing {package}.", flush=True)
                continue
            cython_version = "0.29.37" if name == "fairseq" else "3.0.12"
            pip("install", f"Cython=={cython_version}")
            print(f"Building {package}...", flush=True)
            build = subprocess.run(
                [str(PYTHON), "-m", "pip", "install", "--no-build-isolation",
                 "--no-deps", "--verbose", source],
                env=build_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            )
            with install_log.open("a", encoding="utf-8") as log:
                log.write(build.stdout)
            if build.returncode:
                print(build.stdout[-24000:], flush=True)
                raise RuntimeError(f"Native build failed for {package}. See the compiler output above.")
        report(5, "Installing audio and web interface dependencies")
        uv_install("--no-build-isolation",
            "gradio==3.14.0", "fastapi==0.88.0", "starlette==0.22.0", "pydantic==1.10.2",
            "uvicorn==0.20.0", "httpx==0.23.0", "python-multipart==0.0.5", "websockets==10.4",
            "altair==4.2.2", "jinja2==3.1.4", "pandas==1.5.3", "matplotlib==3.6.2", "pillow==9.5.0",
            "markdown-it-py==2.2.0", "mdit-py-plugins==0.3.3", "linkify-it-py==1.0.3",
            "numpy==1.23.5", "scipy==1.9.3", "numba==0.56.4", "llvmlite==0.39.1", "librosa==0.9.1",
            "scikit-learn==1.2.2", "resampy==0.4.3", "soundfile==0.12.1", "av==12.4.0",
            "faiss-cpu==1.7.4", "praat-parselmouth==0.4.6", "pyworld==0.3.5",
            "tensorboard==2.12.0", "tensorboardX==2.6.2.2", "protobuf==3.20.3", "onnx==1.15.0", "onnxruntime==1.16.3",
            "torchcrepe==0.0.20", "noisereduce==3.0.3", "python-dotenv==1.0.1",
            "tabulate==0.9.0", "pycryptodome==3.23.0", "psutil==6.1.1")
        # Fairseq uses legacy OmegaConf metadata handled by the pinned pip.
        pip("install", "--no-build-isolation",
            "fairseq==0.12.2", "hydra-core==1.0.7", "omegaconf==2.0.6")
        report(6, "Checking dependency versions")
        pip("check")
    else:
        print("Linux environment already installed.")

    report(7, "Testing GPU and audio imports")
    run(PYTHON, "-c", """
import torch
import torchaudio, fairseq, gradio, librosa, av, faiss, pyworld, parselmouth
from infer.modules.vc.modules import VC
from torch.utils.tensorboard import SummaryWriter
from tensorboardX import SummaryWriter as TensorBoardXWriter
assert torch.cuda.is_available(), "PyTorch cannot detect an NVIDIA GPU"
for gpu_id in range(torch.cuda.device_count()):
    print(f"GPU {gpu_id} ready: {torch.cuda.get_device_name(gpu_id)}")
print("PyTorch:", torch.__version__, "| Gradio:", gradio.__version__)
""", cwd=ROOT, env={**runtime_env, "PYTHONPATH": f"{ROOT / 'ilaria'}:{ROOT}"})
    install_marker.touch()
    return runtime_env

started = time.monotonic()
status = (0, "Preparing installation")
finished = Event()

def report(step, message):
    global status
    status = (step, message)

def panel():
    step, message = status
    elapsed = int(time.monotonic() - started)
    minutes, seconds = divmod(elapsed, 60)
    percent = step * 100 // 8
    return HTML(
        '<div style="max-width:640px;padding:16px 20px;border-radius:14px;'
        'background:#33243d;color:#fff1fa;border:1px solid #b68cbf">'
        f'<strong>Installing Ilaria RVC</strong> &nbsp; {percent}%'
        f'<progress aria-label="Installation progress" value="{percent}" max="100" '
        'style="display:block;width:100%;height:18px;accent-color:#ffc0cb;margin:8px 0"></progress>'
        f'<div>{escape(message)} &middot; Elapsed {minutes:02d}:{seconds:02d}</div></div>'
    )

view = display(panel(), display_id=True)

def refresh_display():
    while not finished.wait(1):
        view.update(panel())

timer = Thread(target=refresh_display, daemon=True)
timer.start()
try:
    runtime_env = install_environment(report)
except BaseException:
    report(status[0], "Installation stopped. See the output below.")
    raise
else:
    report(8, "Ready")
finally:
    finished.set()
    timer.join()
    view.update(panel())


## 3. Download base weights

The default downloads the 32 kHz pretrained pair. Add the other rates if you plan to train at 40 or 48 kHz.

In [ ]:
#@title Download base weights
import hashlib
import time
from urllib.request import Request, urlopen

TRAINING_SAMPLE_RATES = ["32k"]  # [] for inference only, or ["32k", "40k", "48k"]
DOWNLOAD_FCPE = True
ASSET_SOURCE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/e6d0c1a17da07c33557852f9dfa2bd44cc75737d"
PRETRAIN_HASHES = {
    "32k": ("2332611297b8d88c7436de8f17ef5f07a2119353e962cd93cda5806d59a1133d",
            "bd7134e7793674c85474d5145d2d982e3c5d8124fc7bb6c20f710ed65808fa8a"),
    "40k": ("3b2c44035e782c4b14ddc0bede9e2f4a724d025cd073f736d4f43708453adfcb",
            "6b6ab091e70801b28e3f41f335f2fc5f3f35c75b39ae2628d419644ec2b0fa09"),
    "48k": ("b5d51f589cc3632d4eae36a315b4179397695042edc01d15312e1bddc2b764a4",
            "2269b73c7a4cf34da09aea99274dabf99b2ddb8a42cbfb065fb3c0aa9a2fc748"),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download(url, destination, expected_hash):
    if destination.is_file() and sha256_file(destination) == expected_hash:
        print(f"{destination.name}: already verified.")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, 4):
        print(f"Downloading {destination.name} (attempt {attempt}/3)...", flush=True)
        try:
            with urlopen(Request(url, headers={"User-Agent": "Ilaria-RVC-Colab/2.4.0"}), timeout=90) as response:
                with temporary.open("wb") as stream:
                    shutil.copyfileobj(response, stream, length=4 * 1024 * 1024)
            if sha256_file(temporary) != expected_hash:
                raise ValueError(f"SHA-256 check failed for {destination.name}.")
            temporary.replace(destination)
            print(f"{destination.name}: ready.")
            return
        except (OSError, ValueError):
            if attempt == 3:
                raise
            time.sleep(2 * attempt)
        finally:
            temporary.unlink(missing_ok=True)

weights = [
    ("hubert_base.pt", "assets/hubert/hubert_base.pt",
     "f54b40fd2802423a5643779c4861af1e9ee9c1564dc9d32f54f20b5ffba7db96"),
    ("rmvpe.pt", "assets/rmvpe/rmvpe.pt",
     "6d62215f4306e3ca278246188607209f09af3dc77ed4232efdd069798c4ec193"),
]
for rate in TRAINING_SAMPLE_RATES:
    for role, digest in zip(("G", "D"), PRETRAIN_HASHES[rate]):
        filename = f"pretrained_v2/f0{role}{rate}.pth"
        weights.append((filename, f"assets/{filename}", digest))
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=3) as executor:
    downloads = [
        executor.submit(download, f"{ASSET_SOURCE}/{filename}", ROOT / relative_destination, digest)
        for filename, relative_destination, digest in dict.fromkeys(weights)
    ]
    for task in downloads:
        task.result()

if DOWNLOAD_FCPE:
    from zipfile import ZipFile
    fcpe_target = ROOT / "assets/pitch/fcpe_runtime/torchfcpe/assets/fcpe_c_v001.pt"
    fcpe_hash = "b9aeaeb673436eeda50ceafd632aa681aa63417e52eae4207503d180c9b10015"
    if not fcpe_target.is_file() or sha256_file(fcpe_target) != fcpe_hash:
        fcpe_wheel = TOOLS / "torchfcpe-0.0.4-py3-none-any.whl"
        fcpe_url = (
            "https://files.pythonhosted.org/packages/15/98/"
            "66da447498a2835b01cb3ae851e1333035a9fbf228e5c6f3e3c4351c06b6/"
            "torchfcpe-0.0.4-py3-none-any.whl"
        )
        try:
            download(fcpe_url, fcpe_wheel, "f042c463d850d76c6f4899a0b84f0b694bb560adf05f4de951097a756d17472d")
            fcpe_target.parent.mkdir(parents=True, exist_ok=True)
            temporary = fcpe_target.with_suffix(".part")
            try:
                with ZipFile(fcpe_wheel) as archive, archive.open("torchfcpe/assets/fcpe_c_v001.pt") as model:
                    with temporary.open("wb") as stream:
                        shutil.copyfileobj(model, stream)
                if sha256_file(temporary) != fcpe_hash:
                    raise ValueError("SHA-256 check failed for FCPE.")
                temporary.replace(fcpe_target)
            finally:
                temporary.unlink(missing_ok=True)
        except (OSError, KeyError, ValueError) as error:
            print(f"FCPE unavailable ({error}). Use RMVPE.")
print("Base weights ready.")


## 4. Start the Web UI

Keep this Colab runtime connected while using the app. Rerun this cell to reopen the embedded view.

In [ ]:
#@title Start Ilaria RVC
import signal
import socket
import time
from urllib.error import URLError
from urllib.request import urlopen
from google.colab import output

def stop_ilaria():
    process = globals().get("ilaria_process")
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait(timeout=5)

log_path = TOOLS / "ilaria-rvc.log"
if globals().get("ilaria_process") is None or ilaria_process.poll() is not None:
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    with log_path.open("wb") as log:
        ilaria_process = subprocess.Popen(
            [str(PYTHON), "-u", "ILARIA_RVC.py", "--pycmd", str(PYTHON),
             "--port", str(port), "--noautoopen"],
            cwd=ROOT, env=runtime_env, stdout=log, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    print("Starting Ilaria RVC…", flush=True)

deadline = time.monotonic() + 150
while time.monotonic() < deadline:
    if ilaria_process.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{port}/config", timeout=3) as response:
            if response.status == 200:
                print("Ilaria RVC is ready.")
                output.serve_kernel_port_as_iframe(port, height=920)
                break
    except (URLError, TimeoutError, OSError):
        pass
    time.sleep(1)
else:
    stop_ilaria()
    raise RuntimeError("Startup timed out. See " + str(log_path))
if ilaria_process.poll() is not None:
    raise RuntimeError("Ilaria RVC exited. Recent log output:\n" +
                       log_path.read_text(encoding="utf-8", errors="replace")[-6000:])


## Export results (optional)

Finish or safely stop training first. Turn on **Include checkpoints** to keep resumable runs. This can create a large ZIP.

In [ ]:
#@title Export models and converted audio
EXPORT = False  #@param {type:"boolean"}
INCLUDE_CHECKPOINTS = False  #@param {type:"boolean"}

if EXPORT:
    from zipfile import ZIP_STORED, ZipFile
    from google.colab import files

    archive = WORK / "Ilaria-RVC-results.zip"
    temporary = archive.with_suffix(".part")
    folders = ["models", "InferOutput"] + (["logs"] if INCLUDE_CHECKPOINTS else [])
    try:
        with ZipFile(temporary, "w", compression=ZIP_STORED, allowZip64=True) as bundle:
            for name in folders:
                for path in (ROOT / name).rglob("*"):
                    if path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(ROOT):
                        bundle.write(path, path.relative_to(ROOT))
        temporary.replace(archive)
    finally:
        temporary.unlink(missing_ok=True)
    print(f"Exported {archive.stat().st_size / 1024**2:.1f} MiB")
    files.download(str(archive))


## Stop the Web UI (optional)

Use **Stop and save** in the Web UI first if training is still running.

In [ ]:
#@title Stop Ilaria RVC
STOP = False  #@param {type:"boolean"}
if STOP and "stop_ilaria" in globals():
    stop_ilaria()
    print("Ilaria RVC stopped.")
